In [11]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "id": "71315d71",
   "metadata": {},
   "source": [
    "# Wine Classification with Machine Learning\n",
    "\n",
    "## Project objective\n",
    "Build a supervised machine-learning model that predicts which of three wine cultivars a sample belongs to, using the 13 chemical measurements in scikit-learn's Wine Recognition dataset.\n",
    "\n",
    "**Dataset:** `sklearn.datasets.load_wine`  \n",
    "**Problem type:** Multiclass classification (3 classes)  \n",
    "**Models:** Logistic Regression (baseline) and Random Forest (comparison)\n"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "08037a5d",
   "metadata": {},
   "source": [
    "## 1. Import libraries\n",
    "\n",
    "We use:\n",
    "- **pandas / NumPy** to inspect and manipulate data.\n",
    "- **scikit-learn** to load the dataset, split the data, build models, and evaluate performance.\n",
    "- **matplotlib / seaborn** to visualize class balance and the confusion matrix."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": None,
   "id": "ea1497f3",
   "metadata": {},
   "outputs": [],
   "source": [
    "import numpy as np\n",
    "import pandas as pd\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "\n",
    "from sklearn.datasets import load_wine\n",
    "from sklearn.model_selection import train_test_split\n",
    "from sklearn.pipeline import Pipeline\n",
    "from sklearn.preprocessing import StandardScaler\n",
    "from sklearn.linear_model import LogisticRegression\n",
    "from sklearn.ensemble import RandomForestClassifier\n",
    "from sklearn.metrics import (\n",
    "    accuracy_score,\n",
    "    classification_report,\n",
    "    confusion_matrix,\n",
    "    ConfusionMatrixDisplay\n",
    ")\n",
    "\n",
    "RANDOM_STATE = 42"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "54f61f59",
   "metadata": {},
   "source": [
    "## 2. Load the Wine dataset\n",
    "\n",
    "`X` contains the 13 input features, while `y` contains the target class labels."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": None,
   "id": "894cd7e5",
   "metadata": {},
   "outputs": [],
   "source": [
    "# Load the dataset from scikit-learn\n",
    "wine = load_wine()\n",
    "\n",
    "# Create a DataFrame so the features are easier to inspect\n",
    "X = pd.DataFrame(wine.data, columns=wine.feature_names)\n",
    "y = pd.Series(wine.target, name=\"wine_class\")\n",
    "\n",
    "# Combine features and target for a quick overview\n",
    "df = X.copy()\n",
    "df[\"wine_class\"] = y\n",
    "\n",
    "print(\"Dataset shape:\", df.shape)\n",
    "print(\"Feature count:\", X.shape[1])\n",
    "print(\"Class names:\", list(wine.target_names))\n",
    "display(df.head())"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "cba1f850",
   "metadata": {},
   "source": [
    "## 3. Explore the data\n",
    "\n",
    "Check for missing values, summary statistics, and the number of samples in each class. This helps us understand the dataset before training."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": None,
   "id": "25b3b8f6",
   "metadata": {},
   "outputs": [],
   "source": [
    "# Check missing values in each column\n",
    "print(\"Missing values per column:\")\n",
    "display(X.isnull().sum())\n",
    "\n",
    "# Summary statistics for the numerical features\n",
    "display(X.describe().T)\n",
    "\n",
    "# Count the number of samples in each target class\n",
    "class_counts = y.value_counts().sort_index()\n",
    "class_counts.index = [wine.target_names[i] for i in class_counts.index]\n",
    "\n",
    "print(\"Class distribution:\")\n",
    "display(class_counts)\n",
    "\n",
    "# Visualize class balance\n",
    "class_counts.plot(kind=\"bar\", rot=0)\n",
    "plt.title(\"Number of Wine Samples per Class\")\n",
    "plt.xlabel(\"Wine class\")\n",
    "plt.ylabel(\"Number of samples\")\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "62e12b16",
   "metadata": {},
   "source": [
    "## 4. Split into training and testing sets\n",
    "\n",
    "We hold out 20% of the samples for final testing. `stratify=y` keeps approximately the same class proportions in both sets. The test set is not used to fit the models."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": None,
   "id": "8a418863",
   "metadata": {},
   "outputs": [],
   "source": [
    "X_train, X_test, y_train, y_test = train_test_split(\n",
    "    X,\n",
    "    y,\n",
    "    test_size=0.20,\n",
    "    random_state=RANDOM_STATE,\n",
    "    stratify=y\n",
    ")\n",
    "\n",
    "print(\"Training samples:\", X_train.shape[0])\n",
    "print(\"Testing samples:\", X_test.shape[0])"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "83e50cd5",
   "metadata": {},
   "source": [
    "## 5. Build and train a Logistic Regression model\n",
    "\n",
    "Logistic Regression is a useful baseline for multiclass classification. The Wine features have different measurement scales, so we place `StandardScaler` and the classifier in a **Pipeline**. This ensures the scaler is fitted only on the training data and helps prevent data leakage.\n",
    "\n",
    "The `lbfgs` solver supports multiclass classification. We increase `max_iter` to give the optimizer enough iterations to converge."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": None,
   "id": "2c12fb00",
   "metadata": {},
   "outputs": [],
   "source": [
    "# Build a pipeline: standardize features, then fit Logistic Regression\n",
    "logistic_model = Pipeline(steps=[\n",
    "    (\"scaler\", StandardScaler()),\n",
    "    (\"classifier\", LogisticRegression(\n",
    "        max_iter=2000,\n",
    "        random_state=RANDOM_STATE\n",
    "    ))\n",
    "])\n",
    "\n",
    "# Train using only the training set\n",
    "logistic_model.fit(X_train, y_train)\n",
    "\n",
    "# Predict the labels of previously unseen test samples\n",
    "logistic_predictions = logistic_model.predict(X_test)\n",
    "\n",
    "print(\"Logistic Regression accuracy:\",\n",
    "      round(accuracy_score(y_test, logistic_predictions), 4))"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "a138f348",
   "metadata": {},
   "source": [
    "## 6. Evaluate Logistic Regression\n",
    "\n",
    "Accuracy gives the overall proportion of correct predictions. The classification report also provides precision, recall, and F1-score for each class. The confusion matrix shows which classes are being confused.\n",
    "\n",
    "Because this is a small dataset, the reported test metrics can vary depending on the train/test split. They should not be treated as a guarantee of future performance."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": None,
   "id": "fb78f782",
   "metadata": {},
   "outputs": [],
   "source": [
    "print(classification_report(\n",
    "    y_test,\n",
    "    logistic_predictions,\n",
    "    target_names=wine.target_names,\n",
    "    zero_division=0\n",
    "))\n",
    "\n",
    "ConfusionMatrixDisplay.from_predictions(\n",
    "    y_test,\n",
    "    logistic_predictions,\n",
    "    display_labels=wine.target_names,\n",
    "    cmap=\"Blues\"\n",
    ")\n",
    "plt.title(\"Logistic Regression - Confusion Matrix\")\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "1688143f",
   "metadata": {},
   "source": [
    "## 7. Train a Random Forest model for comparison\n",
    "\n",
    "Random Forest combines many decision trees. It can learn nonlinear relationships and does not require feature scaling, so this pipeline uses the classifier directly. We set the random seed to make the experiment reproducible."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": None,
   "id": "162a1806",
   "metadata": {},
   "outputs": [],
   "source": [
    "random_forest_model = RandomForestClassifier(\n",
    "    n_estimators=300,\n",
    "    random_state=RANDOM_STATE\n",
    ")\n",
    "\n",
    "random_forest_model.fit(X_train, y_train)\n",
    "rf_predictions = random_forest_model.predict(X_test)\n",
    "\n",
    "print(\"Random Forest accuracy:\",\n",
    "      round(accuracy_score(y_test, rf_predictions), 4))\n",
    "\n",
    "print(classification_report(\n",
    "    y_test,\n",
    "    rf_predictions,\n",
    "    target_names=wine.target_names,\n",
    "    zero_division=0\n",
    "))"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "105bd237",
   "metadata": {},
   "source": [
    "## 8. Compare the models\n",
    "\n",
    "Compare test accuracy for the two models. Accuracy is easy to interpret, but for a fuller evaluation also consider each class's precision, recall, and F1-score from the reports above."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": None,
   "id": "fab742e6",
   "metadata": {},
   "outputs": [],
   "source": [
    "results = pd.DataFrame({\n",
    "    \"Model\": [\"Logistic Regression\", \"Random Forest\"],\n",
    "    \"Test accuracy\": [\n",
    "        accuracy_score(y_test, logistic_predictions),\n",
    "        accuracy_score(y_test, rf_predictions)\n",
    "    ]\n",
    "}).sort_values(\"Test accuracy\", ascending=False)\n",
    "\n",
    "display(results)\n",
    "\n",
    "ax = results.plot(\n",
    "    x=\"Model\",\n",
    "    y=\"Test accuracy\",\n",
    "    kind=\"bar\",\n",
    "    legend=False,\n",
    "    ylim=(0, 1),\n",
    "    rot=0\n",
    ")\n",
    "ax.set_ylabel(\"Accuracy\")\n",
    "ax.set_title(\"Test Accuracy by Model\")\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "68b63421",
   "metadata": {},
   "source": [
    "## 9. Make a prediction for one test example\n",
    "\n",
    "This demonstrates how the trained model can classify an individual sample. We compare the predicted class with the known test label. In a real application, the input would be a new wine sample whose class is not known."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": None,
   "id": "e22729d8",
   "metadata": {},
   "outputs": [],
   "source": [
    "# Select the first row from the test set\n",
    "sample = X_test.iloc[[0]]\n",
    "\n",
    "# Predict using the Logistic Regression pipeline\n",
    "predicted_class = logistic_model.predict(sample)[0]\n",
    "actual_class = y_test.iloc[0]\n",
    "\n",
    "print(\"Predicted class:\", wine.target_names[predicted_class])\n",
    "print(\"Actual class:   \", wine.target_names[actual_class])"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "d7124cec",
   "metadata": {},
   "source": [
    "## 10. Conclusion\n",
    "\n",
    "In this notebook, we:\n",
    "1. Loaded and explored the Wine Recognition dataset.\n",
    "2. Split the data into stratified training and testing sets.\n",
    "3. Trained a scaled Logistic Regression classifier.\n",
    "4. Evaluated it using accuracy, precision, recall, F1-score, and a confusion matrix.\n",
    "5. Trained a Random Forest classifier and compared its test accuracy.\n",
    "\n",
    "The best-performing model on this particular split can be identified from the comparison table. A single train/test split is only an initial estimate; for a more reliable comparison, use stratified cross-validation and consider hyperparameter tuning. The dataset is small and clean, so results may be more optimistic than performance on real-world wine data."
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}

{'cells': [{'cell_type': 'markdown',
   'id': '71315d71',
   'metadata': {},
   'source': ['# Wine Classification with Machine Learning\n',
    '\n',
    '## Project objective\n',
    "Build a supervised machine-learning model that predicts which of three wine cultivars a sample belongs to, using the 13 chemical measurements in scikit-learn's Wine Recognition dataset.\n",
    '\n',
    '**Dataset:** `sklearn.datasets.load_wine`  \n',
    '**Problem type:** Multiclass classification (3 classes)  \n',
    '**Models:** Logistic Regression (baseline) and Random Forest (comparison)\n']},
  {'cell_type': 'markdown',
   'id': '08037a5d',
   'metadata': {},
   'source': ['## 1. Import libraries\n',
    '\n',
    'We use:\n',
    '- **pandas / NumPy** to inspect and manipulate data.\n',
    '- **scikit-learn** to load the dataset, split the data, build models, and evaluate performance.\n',
    '- **matplotlib / seaborn** to visualize class balance and the confusion matrix.']},
  {'cell_type': '